# Randomize yearly datasets into game trials

Assign the 17 datasets from 2000–2016 to 24 trials across three blocks. Every year is used once, seven randomly selected years are used a second time, and the complete trial order is shuffled. The same assignment is used for each matching `items`, `pf`, and `pf_indices` file.

In [1]:
from pathlib import Path
import csv
import random
import shutil

sub_id = 0

# Paths are relative to this notebook/repository root.
DATA_DIR = Path("data")
ITEMS_DIR = DATA_DIR / "items"
PF_DIR = DATA_DIR / "unbiased_pf"
OUTPUT_DIR = DATA_DIR / "game_data"

YEARS = list(range(2000, 2017))
BLOCK_SIZES = {1: 6, 2: 13, 3: 5}
SEED = sub_id  # Change this value to create a different reproducible assignment.


def source_files(year):
    """Return the three source files associated with one year."""
    return {
        "items": ITEMS_DIR / f"items_{year}.csv",
        "pf": PF_DIR / f"pf_{year}.csv",
        "pf_indices": PF_DIR / f"pf_indices_{year}.csv",
    }


# Validate all inputs before writing anything.
missing = [
    path
    for year in YEARS
    for path in source_files(year).values()
    if not path.is_file()
]
if missing:
    raise FileNotFoundError(
        "Missing source files:\n" + "\n".join(str(path) for path in missing)
    )

rng = random.Random(SEED)
# 17 years fill the first 17 slots; seven distinct years fill the remaining slots.
assignments = YEARS + rng.sample(YEARS, sum(BLOCK_SIZES.values()) - len(YEARS))
rng.shuffle(assignments)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = []
assignment_index = 0

for block, trial_count in BLOCK_SIZES.items():
    for trial in range(1, trial_count + 1):
        year = assignments[assignment_index]
        assignment_index += 1

        destinations = {
            "items": OUTPUT_DIR / f"block_{block}_trial_{trial}_{year}.csv",
            "pf": OUTPUT_DIR / f"pf_block_{block}_trial_{trial}_{year}.csv",
            "pf_indices": OUTPUT_DIR / f"pf_indices_block_{block}_trial_{trial}_{year}.csv",
        }

        for kind, source in source_files(year).items():
            shutil.copy2(source, destinations[kind])

        manifest.append({"block": block, "trial": trial, "year": year})

# Save the randomization separately so it is easy to inspect and reproduce.
manifest_path = OUTPUT_DIR / "trial_assignments.csv"
with manifest_path.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["block", "trial", "year"])
    writer.writeheader()
    writer.writerows(manifest)

print(f"Copied {len(manifest) * 3} files to {OUTPUT_DIR}")
print(f"Assignment manifest: {manifest_path}")
manifest

Copied 72 files to data/game_data
Assignment manifest: data/game_data/trial_assignments.csv


[{'block': 1, 'trial': 1, 'year': 2003},
 {'block': 1, 'trial': 2, 'year': 2007},
 {'block': 1, 'trial': 3, 'year': 2013},
 {'block': 1, 'trial': 4, 'year': 2010},
 {'block': 1, 'trial': 5, 'year': 2000},
 {'block': 1, 'trial': 6, 'year': 2005},
 {'block': 2, 'trial': 1, 'year': 2000},
 {'block': 2, 'trial': 2, 'year': 2008},
 {'block': 2, 'trial': 3, 'year': 2002},
 {'block': 2, 'trial': 4, 'year': 2014},
 {'block': 2, 'trial': 5, 'year': 2008},
 {'block': 2, 'trial': 6, 'year': 2007},
 {'block': 2, 'trial': 7, 'year': 2004},
 {'block': 2, 'trial': 8, 'year': 2001},
 {'block': 2, 'trial': 9, 'year': 2012},
 {'block': 2, 'trial': 10, 'year': 2012},
 {'block': 2, 'trial': 11, 'year': 2006},
 {'block': 2, 'trial': 12, 'year': 2004},
 {'block': 2, 'trial': 13, 'year': 2016},
 {'block': 3, 'trial': 1, 'year': 2006},
 {'block': 3, 'trial': 2, 'year': 2013},
 {'block': 3, 'trial': 3, 'year': 2011},
 {'block': 3, 'trial': 4, 'year': 2015},
 {'block': 3, 'trial': 5, 'year': 2009}]

In [6]:
import pandas as pd
import numpy as np

pf = pd.read_csv("data/game_data/pf_block_1_trial_2_2007.csv")

In [7]:
print(np.max(pf, axis=0))
print(np.min(pf, axis=0))

PTS    175.037071
TRB    116.221292
STL     17.942813
BLK     24.026517
FG%    595.900000
dtype: float64
PTS    105.582382
TRB     61.415401
STL      8.149073
BLK      6.351332
FG%    460.700000
dtype: float64


In [10]:
ranges = pd.read_csv("data/game_data/block_1_trial_2_2007_single_solution.csv")
print(ranges)
first_row = ranges.drop(columns="SALARY").iloc[0].to_numpy(dtype=float)
print(first_row)

                                                 PTS  \
0                                         202.895541   
1    [30, 45, 35, 178, 111, 302, 246, 251, 308, 311]   
2                                          56.843783   
3  [153, 168, 299, 189, 227, 230, 331, 180, 271, 19]   

                                                 TRB  \
0                                         123.098337   
1    [239, 271, 47, 163, 93, 25, 145, 157, 230, 299]   
2                                          23.303449   
3  [228, 277, 315, 294, 170, 323, 125, 122, 253, ...   

                                                STL  \
0                                         21.360585   
1   [160, 140, 251, 18, 325, 307, 41, 304, 45, 219]   
2                                          3.312151   
3  [152, 1, 203, 296, 238, 120, 210, 290, 168, 141]   

                                                 BLK  \
0                                          29.985836   
1   [120, 168, 25, 234, 299, 66, 152, 161, 250, 13